# 04 · Catálogo de indicadores (solo lectura)

**Objetivo:** listar qué métricas sirven para **pérdida de bosque** y **expansión agrícola**, por fuente y por año, y calcular las que ya se pueden sacar de MapBiomas.

- Solo lee datos y no escribe archivos.
- La **definición de bosque está pendiente**. Los grupos de la sección 1 son una **propuesta editable**, no una decisión.
- Las métricas de exactitud que publica cada producto no están en nuestros archivos. Hay que buscarlas en su documentación.

**Estado:** notebook nuevo, todavía sin ejecutar.


## 1. Configuración


In [ ]:
from datetime import date, datetime
from pathlib import Path
import hashlib
import io
import json
import re

import ee
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

GEE_PROJECT = "viirs-peru"
ANIO_INI, ANIO_FIN = 2010, 2025  # 15 periodos anuales: 2010 -> 2025

# Bandas por sensor: nombre común -> banda original
BANDAS_LANDSAT = {
    "L5": {"blue": "SR_B1", "green": "SR_B2", "red": "SR_B3", "nir": "SR_B4", "swir1": "SR_B5", "swir2": "SR_B7"},
    "L7": {"blue": "SR_B1", "green": "SR_B2", "red": "SR_B3", "nir": "SR_B4", "swir1": "SR_B5", "swir2": "SR_B7"},
    "L8": {"blue": "SR_B2", "green": "SR_B3", "red": "SR_B4", "nir": "SR_B5", "swir1": "SR_B6", "swir2": "SR_B7"},
    "L9": {"blue": "SR_B2", "green": "SR_B3", "red": "SR_B4", "nir": "SR_B5", "swir1": "SR_B6", "swir2": "SR_B7"},
}
BANDAS_S2 = {"blue": "B2", "green": "B3", "red": "B4", "nir": "B8", "swir1": "B11", "swir2": "B12"}
COLECCIONES_LANDSAT = {
    "L5": "LANDSAT/LT05/C02/T1_L2",
    "L7": "LANDSAT/LE07/C02/T1_L2",
    "L8": "LANDSAT/LC08/C02/T1_L2",
    "L9": "LANDSAT/LC09/C02/T1_L2",
}
COLECCION_S2 = "COPERNICUS/S2_SR_HARMONIZED"
MB_ASSET = "projects/mapbiomas-public/assets/peru/collection4/mapbiomas_peru_collection4_coverage_v1"


def buscar_raiz(inicio=None):
    """Sube por las carpetas hasta encontrar la raíz del proyecto (la que tiene el límite provincial).
    Se usa el GeoPackage como marcador porque puede existir una carpeta data/raw dentro de notebooks/."""
    inicio = Path(inicio or Path.cwd()).resolve()
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg").is_file():
            return carpeta
    raise FileNotFoundError("No encuentro data/raw/Provincia/PROVINCIA.gpkg desde esta ubicación.")


def iniciar_gee(proyecto=GEE_PROJECT):
    try:
        ee.Initialize(project=proyecto)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=proyecto)


def rango_fechas(col):
    """Primera y última fecha de una colección, como texto."""
    d = ee.Dictionary({
        "min": col.aggregate_min("system:time_start"),
        "max": col.aggregate_max("system:time_start"),
    }).getInfo()
    if d["min"] is None:
        return "sin fecha"
    a = pd.to_datetime(d["min"], unit="ms").strftime("%Y-%m-%d")
    b = pd.to_datetime(d["max"], unit="ms").strftime("%Y-%m-%d")
    return f"{a} → {b}"


def md5(ruta):
    """Suma MD5 de un archivo, leído por bloques."""
    h = hashlib.md5()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


iniciar_gee()
RAIZ = buscar_raiz()
GPKG = RAIZ / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg"
gdf = gpd.read_file(GPKG)
prov = gdf[(gdf["ccdd"] == "08") & (gdf["ccpp"] == "01")]
assert len(prov) == 1, "El filtro de provincia no devolvió exactamente 1 registro"
aoi = ee.Geometry(prov.geometry.iloc[0].simplify(0.0001, preserve_topology=True).__geo_interface__)
print("Provincia:", prov["nombprov"].iloc[0], "| ventana:", ANIO_INI, "-", ANIO_FIN)


## 2. Grupos de clases (propuesta editable)

Cada código de MapBiomas se asigna a un grupo del proyecto. Lo que no esté en la lista cae en "otras".


In [ ]:
# PROPUESTA. Ajustar cuando se decida la definición de bosque (01_alcance, pendiente).
GRUPOS = {
    "bosque": [3, 4, 5, 6],   # bosque, bosque seco, manglar, bosque inundable
    "agricola": [21],         # mosaico agropecuario
    "pastos": [12, 15],       # formación herbácea, pasto
    "matorral": [66],         # matorral y otros arbustales
    "urbano": [24],           # infraestructura urbana
    "agua": [31, 33],         # acuicultura, río/lago
}


def grupo_de(codigo):
    for grupo, codigos in GRUPOS.items():
        if codigo in codigos:
            return grupo
    return "otras"


NOMBRE_MB = {
    3: "Bosque", 4: "Bosque seco", 5: "Manglar", 6: "Bosque inundable",
    9: "Plantación forestal", 12: "Formación herbácea", 15: "Pasto (beta)",
    21: "Mosaico agropecuario", 24: "Infraestructura urbana", 31: "Acuicultura",
    33: "Río, lago u océano", 66: "Matorral y otros arbustales",
    68: "Otra área natural sin vegetación", 82: "Herbazal inundable altoandino",
}
mb = ee.Image(MB_ASSET)
print("Grupos propuestos:", GRUPOS)


## 3. Área por clase y año (MapBiomas)

Área en km² dentro de la provincia, por código de clase.


In [ ]:
def area_por_clase(anio):
    clase = mb.select(f"classification_{anio}").clip(aoi)
    grupos = (ee.Image.pixelArea().addBands(clase)
              .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName="codigo"),
                            geometry=aoi, scale=30, maxPixels=1e10, bestEffort=True)
              .get("groups").getInfo())
    return {int(g["codigo"]): g["sum"] / 1e6 for g in grupos}


areas = {anio: area_por_clase(anio) for anio in range(ANIO_INI, ANIO_FIN + 1)}
tabla_area = pd.DataFrame(areas).T
tabla_area.columns = [f"{c} {NOMBRE_MB.get(c, '')}".strip() for c in tabla_area.columns]
tabla_area.round(2)


## 4. Transiciones entre años consecutivos (MapBiomas)

Para cada periodo se cruza la clase de un año con la del siguiente. Así se obtiene cuánta área pasó de un grupo a otro.

Se usa código = clase inicial × 100 + clase final. Cada par de clases queda como un número único.


In [ ]:
PIXEL_KM2 = 30 * 30 / 1e6  # un píxel de 30 m


def histograma_transicion(anio_a, anio_b):
    a = mb.select(f"classification_{anio_a}").clip(aoi)
    b = mb.select(f"classification_{anio_b}").clip(aoi)
    par = a.multiply(100).add(b).rename("par")
    hist = (par.reduceRegion(reducer=ee.Reducer.frequencyHistogram(), geometry=aoi,
                             scale=30, maxPixels=1e10, bestEffort=True)
            .get("par").getInfo())
    return {int(float(k)): v for k, v in hist.items()}


filas = []
for anio in range(ANIO_INI, ANIO_FIN):
    hist = histograma_transicion(anio, anio + 1)
    acum = {
        "pérdida de bosque (bosque → no bosque)": 0.0,
        "expansión agrícola (no agrícola → agrícola)": 0.0,
        "expansión de bosque (no bosque → bosque)": 0.0,
    }
    for par, pixeles in hist.items():
        c1, c2 = divmod(par, 100)
        km2 = pixeles * PIXEL_KM2
        g1, g2 = grupo_de(c1), grupo_de(c2)
        if g1 == "bosque" and g2 != "bosque":
            acum["pérdida de bosque (bosque → no bosque)"] += km2
        if g1 != "agricola" and g2 == "agricola":
            acum["expansión agrícola (no agrícola → agrícola)"] += km2
        if g1 != "bosque" and g2 == "bosque":
            acum["expansión de bosque (no bosque → bosque)"] += km2
    filas.append({"periodo": f"{anio} → {anio + 1}", **{k: round(v, 2) for k, v in acum.items()}})

indicadores_mb = pd.DataFrame(filas)
area_aoi = aoi.area().divide(1e6).getInfo()
for col in ["pérdida de bosque (bosque → no bosque)", "expansión agrícola (no agrícola → agrícola)",
            "expansión de bosque (no bosque → bosque)"]:
    indicadores_mb[col + " %"] = (100 * indicadores_mb[col] / area_aoi).round(2)

print(f"Área de la provincia: {area_aoi:.1f} km²")
indicadores_mb


## 5. Métricas de GEOBOSQUES (lo que trae su JSON)

Se buscan en el JSON local las rutas que suenan a pérdida, bosque, área o año. Así sabemos si el producto ya trae estos indicadores.


In [ ]:
def rutas_hojas(obj, ruta=""):
    """Recorre el JSON y entrega cada hoja como (ruta, valor)."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from rutas_hojas(v, f"{ruta}.{k}" if ruta else k)
    elif isinstance(obj, list):
        for i, v in enumerate(obj[:3]):
            yield from rutas_hojas(v, f"{ruta}[{i}]")
    else:
        yield ruta, obj


ruta_json = RAIZ / "data" / "raw" / "GEOBOSQUES" / "info_bosque_humedo_2025.json"
if ruta_json.exists():
    info_gb = json.loads(ruta_json.read_text(encoding="utf-8"))
    hojas = list(rutas_hojas(info_gb))
    candidatas = [(r, v) for r, v in hojas if re.search(r"perd|bosq|area|superf|anio|year|ha", r, re.I)]
    pd.DataFrame(candidatas[:80], columns=["ruta", "valor"])
else:
    print("No encuentro", ruta_json)


## 6. Catálogo de indicadores

Lista de métricas posibles. **Estado** indica si ya se calculó aquí, si la fuente la trae, o si falta.


In [ ]:
catalogo = pd.DataFrame([
    ["Área por clase y año", "MapBiomas C4", "referencia", "2010–2025", "km²", "píxeles × 900 m² por clase", "calculado (sección 3)"],
    ["Pérdida de bosque", "MapBiomas C4", "referencia", "por periodo", "km² y %", "bosque en t → no bosque en t+1", "calculado (sección 4)"],
    ["Expansión agrícola", "MapBiomas C4", "referencia", "por periodo", "km² y %", "no agrícola en t → agrícola en t+1", "calculado (sección 4)"],
    ["Expansión de bosque (regeneración)", "MapBiomas C4", "referencia", "por periodo", "km² y %", "no bosque en t → bosque en t+1", "calculado (sección 4)"],
    ["Matriz completa de transiciones", "MapBiomas C4", "referencia", "por periodo", "km²", "todos los pares de clases", "calculado en el histograma; ampliar a tabla"],
    ["Pérdida de bosque 2001–2025", "GEOBOSQUES", "referencia", "2001–2025", "ha", "por confirmar en el JSON", "pendiente (sección 5)"],
    ["Bosque remanente 2025", "GEOBOSQUES", "referencia", "2025", "ha", "por confirmar en el JSON", "pendiente (sección 5)"],
    ["Exactitud global, productor y usuario", "MapBiomas, GEOBOSQUES", "referencia", "—", "%", "publicada en la documentación del producto", "pendiente de buscar"],
    ["Área por clase", "WorldCover", "referencia", "2020, 2021", "km²", "histograma por clase", "disponible en GEE"],
    ["Altitud media", "SRTM", "factor", "fijo", "m", "promedio sobre la provincia", "disponible en GEE"],
    ["Distancia a vías", "OSM", "factor", "fijo (2026)", "m", "distancia al tramo más cercano", "pendiente de calcular"],
    ["Índices NDVI, EVI y NBR", "Landsat / Sentinel-2", "propio (reglas)", "por año", "adimensional", "sobre el compuesto anual", "pendiente: elegir en 05"],
    ["Mapa de clases", "modelo de computer vision", "propio", "por año", "clase", "salida del modelo", "pendiente (línea 2)"],
    ["Tasa de cambio por periodo", "propio (cierre)", "comparación", "por periodo", "%/año", "derivada de las métricas anteriores", "se calcula al final"],
], columns=["indicador", "fuente", "tipo", "años", "unidad", "definición", "estado"])
catalogo
